<a href="https://colab.research.google.com/github/jiayujune/ai-smart-kitchen/blob/main/AI_Recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import ast
import re

In [2]:
#extracting data from zip file
import zipfile
import os

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall()

print(os.listdir())

['.config', 'RAW_recipes.csv', 'archive.zip', 'sample_data']


In [3]:
#reading data from extracted zip file
df = pd.read_csv("RAW_recipes.csv")
print(df.shape)
df.head()

(231637, 12)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [4]:
import ast
df["ingredients"] = df["ingredients"].apply(ast.literal_eval)
df["steps"] = df["steps"].apply(ast.literal_eval)
df["nutrition"] = df["nutrition"].apply(ast.literal_eval)
df["tags"] = df["tags"].apply(ast.literal_eval)

In [5]:
df["calories"] = df["nutrition"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

df[["name", "calories"]].head()

,name,calories
0,arriba baked winter squash mexican style,51.5
1,a bit different breakfast pizza,173.4
2,all in the kitchen chili,269.8
3,alouette potatoes,368.1
4,amish tomato ketchup for canning,352.9


In [6]:
def clean_ingredients(ingredients):
    return [str(i).lower().strip() for i in ingredients]

df["ingredients"] = df["ingredients"].apply(clean_ingredients)

df[["name", "ingredients"]].head()

,name,ingredients
0,arriba baked winter squash mexican style,"[winter squash, mexican seasoning, mixed spice..."
1,a bit different breakfast pizza,"[prepared pizza crust, sausage patty, eggs, mi..."
2,all in the kitchen chili,"[ground beef, yellow onions, diced tomatoes, t..."
3,alouette potatoes,"[spreadable cheese with garlic and herbs, new ..."
4,amish tomato ketchup for canning,"[tomato juice, apple cider vinegar, sugar, sal..."


In [7]:
non_veg_keywords = {
    "chicken", "beef", "mutton", "fish", "prawn", "shrimp", "egg",
    "turkey", "bacon", "ham", "pork", "sausage", "lamb", "meat", "salmon", "tuna"
}

def get_food_type(ingredients):
    ingredient_text = " ".join(ingredients)
    for word in non_veg_keywords:
        if word in ingredient_text:
            return "non-veg"
    return "veg"

df["food_type"] = df["ingredients"].apply(get_food_type)

df[["name", "food_type"]].head()

,name,food_type
0,arriba baked winter squash mexican style,veg
1,a bit different breakfast pizza,non-veg
2,all in the kitchen chili,non-veg
3,alouette potatoes,veg
4,amish tomato ketchup for canning,veg


In [8]:
df = df[["id", "name", "ingredients", "steps", "calories", "food_type", "tags"]]
print(df.shape)
df.head()

(231637, 7)


,id,name,ingredients,steps,calories,food_type,tags
0,137739,arriba baked winter squash mexican style,"[winter squash, mexican seasoning, mixed spice...","[make a choice and proceed with recipe, depend...",51.5,veg,"[60-minutes-or-less, time-to-make, course, mai..."
1,31490,a bit different breakfast pizza,"[prepared pizza crust, sausage patty, eggs, mi...","[preheat oven to 425 degrees f, press dough in...",173.4,non-veg,"[30-minutes-or-less, time-to-make, course, mai..."
2,112140,all in the kitchen chili,"[ground beef, yellow onions, diced tomatoes, t...","[brown ground beef in large pot, add chopped o...",269.8,non-veg,"[time-to-make, course, preparation, main-dish,..."
3,59389,alouette potatoes,"[spreadable cheese with garlic and herbs, new ...",[place potatoes in a large pot of lightly salt...,368.1,veg,"[60-minutes-or-less, time-to-make, course, mai..."
4,44061,amish tomato ketchup for canning,"[tomato juice, apple cider vinegar, sugar, sal...","[mix all ingredients& boil for 2 1 / 2 hours ,...",352.9,veg,"[weeknight, time-to-make, course, main-ingredi..."


In [9]:
def recommend_recipes(user_ingredients, food_pref="veg", allergies=None, max_calories=None, top_n=5):
    if allergies is None:
        allergies = []

    user_ingredients = [i.lower().strip() for i in user_ingredients]
    allergies = [a.lower().strip() for a in allergies]

    filtered_df = df.copy()

    # Veg / non-veg filter
    if food_pref == "veg":
        filtered_df = filtered_df[filtered_df["food_type"] == "veg"]

    # Allergy filter
    def has_allergy(recipe_ingredients):
        return any(allergy in recipe_ingredients for allergy in allergies)

    filtered_df = filtered_df[~filtered_df["ingredients"].apply(has_allergy)]

    # Calorie filter
    if max_calories is not None:
        filtered_df = filtered_df[filtered_df["calories"] <= max_calories]

    # Match score
    def compute_scores(recipe_ingredients):
        matched = len(set(user_ingredients).intersection(set(recipe_ingredients)))
        missing = len(set(recipe_ingredients) - set(user_ingredients))
        return pd.Series([matched, missing])

    filtered_df[["matched_count", "missing_count"]] = filtered_df["ingredients"].apply(compute_scores)

    # Keep recipes with at least one match
    filtered_df = filtered_df[filtered_df["matched_count"] > 0]

    # Sort by best match
    filtered_df = filtered_df.sort_values(
        by=["matched_count", "missing_count"],
        ascending=[False, True]
    )

    return filtered_df.head(top_n)

In [10]:
user_ingredients = ["tomato", "onion", "garlic", "cheese", "potato"]
allergies = ["peanut"]

results = recommend_recipes(
    user_ingredients=user_ingredients,
    food_pref="veg",
    allergies=allergies,
    max_calories=500,
    top_n=5
)

results[["name", "ingredients", "calories", "matched_count", "missing_count"]]

,name,ingredients,calories,matched_count,missing_count
40443,cheesy hash browns for one,"[onion, cheese, potato]",23.1,3,0
92911,glen s potato bake,"[potato, salt & pepper, onion, cream, cheese]",266.5,3,2
172830,refried beans the easy but tasty way,"[black beans, olive oil, onion, garlic, salt a...",161.6,3,3
221823,vegetable scrabble,"[carrot, potato, cheese, plain yogurt, onion, ...",1.2,3,3
161698,pole montagnarde alpine hash,"[potato, cheese, lardons, onion, pepper, oil, ...",313.9,3,4


In [11]:
def get_missing_ingredients(recipe_ingredients, user_ingredients):
    return list(set(recipe_ingredients) - set(user_ingredients))

results = results.copy()
results["missing_ingredients"] = results["ingredients"].apply(
    lambda x: get_missing_ingredients(x, [i.lower().strip() for i in user_ingredients])
)

results[["name", "missing_ingredients", "calories"]]

,name,missing_ingredients,calories
40443,cheesy hash browns for one,[],23.1
92911,glen s potato bake,"[cream, salt & pepper]",266.5
172830,refried beans the easy but tasty way,"[salt and pepper, olive oil, black beans]",161.6
221823,vegetable scrabble,"[carrot, butter, plain yogurt]",1.2
161698,pole montagnarde alpine hash,"[pepper, lardons, oil, dry white wine]",313.9


In [12]:
!pip install gradio -q

In [16]:
import gradio as gr
import pandas as pd
import time

# =========================================================
# Helper Functions
# =========================================================

def inventory_to_html(inventory):
    if not inventory:
        return """
        <div style="
            background:#f8fafc;
            border:1px solid #e2e8f0;
            border-radius:16px;
            padding:16px;
            color:#334155;
            font-weight:600;
        ">
            No ingredients left.
        </div>
        """

    chips = "".join([
        f"""
        <span style="
            display:inline-block;
            background:#dbeafe;
            color:#1d4ed8;
            padding:8px 14px;
            border-radius:999px;
            margin:6px;
            font-size:14px;
            font-weight:700;
        ">{item}</span>
        """
        for item in inventory
    ])

    return f"""
    <div style="
        background:white;
        border:1px solid #e5e7eb;
        border-radius:18px;
        padding:14px;
        box-shadow:0 8px 20px rgba(15,23,42,0.05);
    ">
        {chips}
    </div>
    """


def render_recipe_cards(results_df, inventory):
    if results_df.empty:
        return """
        <div style="
            background:#f8fafc;
            border:1px solid #e5e7eb;
            border-radius:16px;
            padding:18px;
            color:#334155;
            font-weight:600;
        ">
            No matching recipes found.
        </div>
        """

    inventory_set = set([i.lower().strip() for i in inventory])
    cards_html = """<div class="recipe-card-grid">"""

    for idx, (_, row) in enumerate(results_df.iterrows(), start=1):
        recipe_ingredients = row["ingredients"] if isinstance(row["ingredients"], list) else []
        matched = list(set(recipe_ingredients).intersection(inventory_set))
        missing = list(set(recipe_ingredients) - inventory_set)

        steps_preview = ""
        if isinstance(row["steps"], list):
            steps_preview = "<br>".join([f"• {s}" for s in row["steps"][:2]])

        score = 0
        if len(recipe_ingredients) > 0:
            score = round(len(matched) / len(recipe_ingredients), 3)

        cards_html += f"""
        <div class="recipe-card">
            <div class="recipe-number">Recipe #{idx}</div>
            <div class="recipe-title">{row['name']}</div>

            <div class="badge-row">
                <span class="badge badge-blue">Score: {score}</span>
                <span class="badge badge-indigo">Matched: {len(matched)}</span>
                <span class="badge badge-cyan">Missing: {len(missing)}</span>
                <span class="badge badge-teal">Calories: {row['calories']}</span>
            </div>

            <div class="recipe-section">
                <div class="recipe-section-title">Matched Ingredients</div>
                <ul>
                    {''.join([f'<li>{x}</li>' for x in matched[:8]]) if matched else '<li>None</li>'}
                </ul>
            </div>

            <div class="recipe-section">
                <div class="recipe-section-title">Missing Ingredients</div>
                <ul>
                    {''.join([f'<li>{x}</li>' for x in missing[:8]]) if missing else '<li>None</li>'}
                </ul>
            </div>

            <div class="recipe-section">
                <div class="recipe-section-title">Steps Preview</div>
                <div class="steps-preview">{steps_preview if steps_preview else 'No steps available'}</div>
            </div>
        </div>
        """

    cards_html += "</div>"
    return cards_html


# =========================================================
# Overlay + Page Flow Functions
# =========================================================

def show_loading_overlay():
    overlay_html = """
    <div class="loading-overlay">
        <div class="loader-wrapper">
          <span class="loader-letter">G</span>
          <span class="loader-letter">e</span>
          <span class="loader-letter">n</span>
          <span class="loader-letter">e</span>
          <span class="loader-letter">r</span>
          <span class="loader-letter">a</span>
          <span class="loader-letter">t</span>
          <span class="loader-letter">i</span>
          <span class="loader-letter">n</span>
          <span class="loader-letter">g</span>

          <div class="loader"></div>
        </div>
    </div>
    """
    return gr.update(value=overlay_html, visible=True)


def generate_results(ingredients_text, food_pref, allergies_text, max_calories):
    time.sleep(2.2)

    inventory = [i.strip().lower() for i in ingredients_text.split(",") if i.strip()]
    allergies = [a.strip().lower() for a in allergies_text.split(",") if a.strip()]

    results = recommend_recipes(
        user_ingredients=inventory,
        food_pref=food_pref,
        allergies=allergies,
        max_calories=max_calories if max_calories else None,
        top_n=6
    )

    inventory_html = inventory_to_html(inventory)
    cards_html = render_recipe_cards(results, inventory)
    shown_results = results.to_dict(orient="records")

    return (
        gr.update(visible=False),   # input page
        gr.update(visible=True),    # results page
        gr.update(value="", visible=False),  # hide overlay
        inventory_html,
        cards_html,
        shown_results,
        inventory
    )


def cook_selected_recipe(recipe_number, current_inventory, shown_results, food_pref):
    if not shown_results:
        return (
            inventory_to_html(current_inventory),
            """
            <div style="
                background:#fff7ed;
                color:#9a3412;
                border:1px solid #fdba74;
                border-radius:16px;
                padding:16px;
                font-weight:700;
            ">
                Please generate recipes first.
            </div>
            """,
            shown_results,
            current_inventory
        )

    try:
        idx = int(recipe_number) - 1
    except:
        return (
            inventory_to_html(current_inventory),
            """
            <div style="
                background:#fef2f2;
                color:#b91c1c;
                border:1px solid #fecaca;
                border-radius:16px;
                padding:16px;
                font-weight:700;
            ">
                Enter a valid recipe number.
            </div>
            """,
            shown_results,
            current_inventory
        )

    if idx < 0 or idx >= len(shown_results):
        return (
            inventory_to_html(current_inventory),
            """
            <div style="
                background:#fef2f2;
                color:#b91c1c;
                border:1px solid #fecaca;
                border-radius:16px;
                padding:16px;
                font-weight:700;
            ">
                Recipe number out of range.
            </div>
            """,
            shown_results,
            current_inventory
        )

    selected_recipe = shown_results[idx]
    recipe_ingredients = [str(i).lower().strip() for i in selected_recipe["ingredients"]]

    updated_inventory = current_inventory.copy()

    for ing in recipe_ingredients:
        if ing in updated_inventory:
            updated_inventory.remove(ing)

    updated_results = recommend_recipes(
        user_ingredients=updated_inventory,
        food_pref=food_pref,
        allergies=[],
        max_calories=None,
        top_n=6
    )

    updated_inventory_html = inventory_to_html(updated_inventory)
    updated_cards_html = render_recipe_cards(updated_results, updated_inventory)
    updated_shown_results = updated_results.to_dict(orient="records")

    return updated_inventory_html, updated_cards_html, updated_shown_results, updated_inventory


def back_to_input():
    return (
        gr.update(visible=True),
        gr.update(visible=False)
    )


def clear_all():
    return "", "veg", "", 500


# =========================================================
# CSS
# =========================================================

custom_css = """
body {
    background: url("/content/tomatoes.jpg") center/cover no-repeat;
}

.gradio-container {
    max-width: 1100px !important;
    margin: 0 auto !important;
    padding: 20px !important;
    font-family: Inter, Arial, sans-serif;
}

.top-card, .input-card, .section-card {
    background: rgba(255,255,255,0.96);
    border-radius: 22px;
    border: 1px solid #e5e7eb;

}

.top-card {
    position: relative;
    padding: 28px;
    margin-bottom: 18px;
    border-radius: 22px;
    overflow: hidden;
    background: #ffffff;  /* pure white */

}



/* Content stays above */
.top-card * {
    position: relative;
    z-index: 2;
}

/* Make text white */
.page-title {
    color: blue !important;
}

.page-subtitle {
    color: #e2e8f0 !important;
}

.input-card {
    padding: 18px;
    margin-bottom: 18px;
}

.section-card {
    padding: 18px;
    margin-bottom: 18px;
}

.page-title {
    font-size: 22px;
    font-weight: 800;
    color: #0f172a;
    margin-bottom: 10px;
}

.page-subtitle {
    color: #64748b;
    font-size: 14px;
    line-height: 1.8;
}

.section-title {
    font-size: 22px;
    font-weight: 800;
    color: #0f172a;
    margin-bottom: 14px;
}

textarea, input {
    border-radius: 14px !important;
    border: 1px solid #cbd5e1 !important;
}

button {
    border-radius: 14px !important;
    font-weight: 700 !important;
}

/* recipe cards */
.recipe-card-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 18px;
}

.recipe-card {
    background: white;
    border: 1px solid #dbe7ff;
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0 10px 24px rgba(30,41,59,0.08);
}

.recipe-number {
    font-size: 13px;
    color: #64748b;
    font-weight: 700;
    margin-bottom: 8px;
}

.recipe-title {
    font-size: 22px;
    font-weight: 800;
    color: #1e3a8a;
    line-height: 1.3;
    margin-bottom: 12px;
}

.badge-row {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-bottom: 14px;
}

.badge {
    padding: 6px 10px;
    border-radius: 999px;
    font-size: 13px;
    font-weight: 700;
}

.badge-blue { background:#e0f2fe; color:#0369a1; }
.badge-indigo { background:#dbeafe; color:#1d4ed8; }
.badge-cyan { background:#eff6ff; color:#2563eb; }
.badge-teal { background:#ecfeff; color:#0f766e; }

.recipe-section {
    margin-bottom: 12px;
}

.recipe-section-title {
    font-weight: 800;
    color: #334155;
    margin-bottom: 6px;
}

.recipe-card ul {
    margin: 0;
    padding-left: 18px;
    color: #475569;
}

.steps-preview {
    color: #475569;
    line-height: 1.6;
}

.helper-text {
    font-size: 13px;
    color: #94a3b8;
    margin-top: 8px;
}

/* full-screen dimmed overlay */
.loading-overlay {
    position: fixed;
    inset: 0;
    width: 100vw;
    height: 100vh;
    background: rgba(2, 6, 23, 0.58);
    backdrop-filter: blur(18px) saturate(120%);
    -webkit-backdrop-filter: blur(18px) saturate(120%);
    display: flex;
    align-items: center;
    justify-content: center;
    z-index: 999999;
    animation: overlayFadeIn 0.35s ease;
}

@keyframes overlayFadeIn {
    from {
        opacity: 0;
    }
    to {
        opacity: 1;
    }
}

.loader-wrapper {
    position: relative;
    display: flex;
    align-items: center;
    justify-content: center;
    width: 240px;
    height: 240px;
    font-family: "Inter", sans-serif;
    font-size: 1.25em;
    font-weight: 300;
    color: white;
    border-radius: 50%;
    background-color: transparent;
    user-select: none;
    transform: scale(1.12);
    filter: drop-shadow(0 0 30px rgba(173, 95, 255, 0.45));
    animation: wrapperPulse 2.2s ease-in-out infinite;
}

@keyframes wrapperPulse {
    0%, 100% {
        transform: scale(1.08);
    }
    50% {
        transform: scale(1.14);
    }
}

.loader {
    position: absolute;
    inset: 0;
    border-radius: 50%;
    background-color: transparent;
    animation: loaderRotate 2.4s linear infinite;
    z-index: 0;
}

@keyframes loaderRotate {
    0% {
        transform: rotate(90deg);
        box-shadow:
            0 10px 20px 0 rgba(255,255,255,0.95) inset,
            0 20px 30px 0 rgba(173,95,255,0.95) inset,
            0 60px 60px 0 rgba(71,30,236,0.95) inset;
    }
    50% {
        transform: rotate(270deg);
        box-shadow:
            0 10px 20px 0 rgba(255,255,255,0.95) inset,
            0 20px 10px 0 rgba(214,10,71,0.85) inset,
            0 40px 60px 0 rgba(49,30,128,0.95) inset;
    }
    100% {
        transform: rotate(450deg);
        box-shadow:
            0 10px 20px 0 rgba(255,255,255,0.95) inset,
            0 20px 30px 0 rgba(173,95,255,0.95) inset,
            0 60px 60px 0 rgba(71,30,236,0.95) inset;
    }
}

.loader-letter {
    display: inline-block;
    opacity: 0.38;
    transform: translateY(0);
    animation: loaderLetterAnim 2s infinite;
    z-index: 1;
    border: none;
    font-size: 1.2rem;
    color: rgba(255,255,255,0.78);
    text-shadow: 0 0 12px rgba(255,255,255,0.08);
}

.loader-letter:nth-child(1) { animation-delay: 0s; }
.loader-letter:nth-child(2) { animation-delay: 0.1s; }
.loader-letter:nth-child(3) { animation-delay: 0.2s; }
.loader-letter:nth-child(4) { animation-delay: 0.3s; }
.loader-letter:nth-child(5) { animation-delay: 0.4s; }
.loader-letter:nth-child(6) { animation-delay: 0.5s; }
.loader-letter:nth-child(7) { animation-delay: 0.6s; }
.loader-letter:nth-child(8) { animation-delay: 0.7s; }
.loader-letter:nth-child(9) { animation-delay: 0.8s; }
.loader-letter:nth-child(10) { animation-delay: 0.9s; }

@keyframes loaderLetterAnim {
    0%, 100% {
        opacity: 0.35;
        transform: translateY(0) scale(1);
    }
    20% {
        opacity: 1;
        transform: translateY(-1px) scale(1.12);
    }
    40% {
        opacity: 0.68;
        transform: translateY(0) scale(1);
    }
}

@media (max-width: 768px) {
    .loader-wrapper {
        width: 180px;
        height: 180px;
        font-size: 1rem;
        transform: scale(1.02);
    }

    .loader-letter {
        font-size: 1rem;
    }
}

/* responsive */
@media (max-width: 1024px) {
    .recipe-card-grid {
        grid-template-columns: repeat(2, minmax(0, 1fr));
    }
}

@media (max-width: 768px) {
    .gradio-container {
        padding: 12px !important;
    }

    .top-card, .input-card, .section-card {
        padding: 14px;
        border-radius: 16px;
    }

    .page-title, .section-title {
        font-size: 20px;
    }

    .recipe-card-grid {
        grid-template-columns: 1fr;
    }

    button {
        width: 100% !important;
    }

    .loader-wrapper {
        width: 170px;
        height: 170px;
        font-size: 1rem;
        transform: scale(1.2);
        filter: drop-shadow(0 0 25px rgba(173, 95, 255, 0.6));
    }
}
"""

# =========================================================
# UI
# =========================================================

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
    shown_results_state = gr.State([])
    inventory_state = gr.State([])

    # PAGE 1
    with gr.Column(visible=True) as input_page:
        gr.Markdown("""
        <div class="top-card">
            <div class="page-title">AI Smart Kitchen Recommender</div>
            <div class="page-subtitle">
                Enter the ingredients currently available in your kitchen, and the system will recommend recipes
                based on ingredient matching, missing items, and calorie information.
            </div>
        </div>
        """)

        with gr.Column(elem_classes="input-card"):
            ingredients_input = gr.Textbox(
                label="Enter ingredients",
                placeholder="milk, eggs, butter, flour"
            )

            with gr.Row():
                food_pref_input = gr.Radio(
                    ["veg", "non-veg"],
                    label="Food Preference",
                    value="veg"
                )
                calories_input = gr.Number(
                    label="Maximum Calories",
                    value=500
                )

            allergies_input = gr.Textbox(
                label="Allergies",
                placeholder="peanut, milk"
            )

            with gr.Row():
                submit_btn = gr.Button("Get Recommendations", variant="primary")
                clear_btn = gr.Button("Clear")

            gr.Markdown('<div class="helper-text">Example: egg, tomato, rice, onion</div>')

    # PAGE 2
    with gr.Column(visible=False) as results_page:
        with gr.Row():
            back_btn = gr.Button("← Back to Input Page")

        gr.Markdown('<div class="section-title">Current Ingredients</div>')
        inventory_output = gr.HTML()

        gr.Markdown('<div class="section-title">Recommendation Results</div>')
        cards_output = gr.HTML()

        with gr.Column(elem_classes="section-card"):
            recipe_choice = gr.Number(label="Recipe Number to Cook", value=1)
            cook_btn = gr.Button("Cook Selected Recipe")

    # OVERLAY
    loading_overlay = gr.HTML(value="", visible=False)

    # actions
    submit_btn.click(
        fn=show_loading_overlay,
        inputs=[],
        outputs=[loading_overlay],
        queue=False
    ).then(
        fn=generate_results,
        inputs=[ingredients_input, food_pref_input, allergies_input, calories_input],
        outputs=[input_page, results_page, loading_overlay, inventory_output, cards_output, shown_results_state, inventory_state]
    )

    cook_btn.click(
        fn=cook_selected_recipe,
        inputs=[recipe_choice, inventory_state, shown_results_state, food_pref_input],
        outputs=[inventory_output, cards_output, shown_results_state, inventory_state]
    )

    back_btn.click(
        fn=back_to_input,
        inputs=[],
        outputs=[input_page, results_page]
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[ingredients_input, food_pref_input, allergies_input, calories_input]
    )

demo.launch(share=True)

/tmp/ipykernel_1406/3141327732.py:617: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_1406/3141327732.py:617: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b16368809a2a8641cd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
